In [1]:
import celltypist
from celltypist import models
from scipy.stats import pearsonr
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import concurrent.futures

In [2]:
expr_path = '/data/scRNA/ABCA/AIBS/AWS/expression_matrices/MERFISH-C57BL6J-638850/20230830/C57BL6J-638850-raw-wmeta.h5ad'
adata = sc.read_h5ad(expr_path)
adata

AnnData object with n_obs × n_vars = 3938808 × 550
    obs: 'brain_section_label', 'cluster_alias', 'average_correlation_score', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'x', 'y', 'z', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color'
    var: 'gene_symbol', 'transcript_identifier'
    uns: 'accessed_on', 'src'

In [5]:
adata_section = adata[adata.obs['brain_section_label'] == 'C57BL6J-638850.38'].copy()
adata_section

AnnData object with n_obs × n_vars = 120186 × 550
    obs: 'brain_section_label', 'cluster_alias', 'average_correlation_score', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'x', 'y', 'z', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color'
    var: 'gene_symbol', 'transcript_identifier'
    uns: 'accessed_on', 'src'

In [6]:
del adata

In [8]:
# models.download_model_index()

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 54


In [9]:
# models.download_models(model="Mouse_Whole_Brain.pkl")

📂 Storing models in /home/momo/.celltypist/data/models
💾 Total models to download: 1
💾 Downloading model [1/1]: Mouse_Whole_Brain.pkl


In [11]:
# models.models_description().head()

👉 Detailed model information can be found at `https://www.celltypist.org/models`


,model,description
0,Immune_All_Low.pkl,immune sub-populations combined from 20 tissue...
1,Immune_All_High.pkl,immune populations combined from 20 tissues of...
2,Adult_COVID19_PBMC.pkl,peripheral blood mononuclear cell types from C...
3,Adult_CynomolgusMacaque_Hippocampus.pkl,cell types from the hippocampus of adult cynom...
4,Adult_Human_MTG.pkl,cell types and subtypes (10x-based) from the a...


In [7]:
model = models.Model.load(model="Mouse_Whole_Brain.pkl")
model

CellTypist model with 334 cell types and 5596 features
    date: 2024-04-30 17:19:37.721313
    details: cell types from the whole adult mouse brain
    source: https://doi.org/10.1038/s41586-023-06812-z
    version: v1
    cell types: 001 CLA-EPd-CTX Car3 Glut, 002 IT EP-CLA Glut, ..., 338 Lymphoid NN
    features: Xkr4, Rgs20, ..., mt-Cytb

In [8]:
# Process data before annotating
sc.pp.normalize_total(adata_section, target_sum=1e4)  # Normalize counts per cell
sc.pp.log1p(adata_section)  # Log-transform the data

In [9]:
adata_section.var.index = adata_section.var["gene_symbol"]  # Replace Ensembl IDs with gene symbols
# adata.var = adata.var[~adata.var.index.str.startswith("Blank")]  # Remove Blank genes

In [20]:
predictions = celltypist.annotate(
    adata_section,
    model = "Mouse_Whole_Brain.pkl",
    use_GPU = False,
    majority_voting = True)

🔬 Input data has 120186 cells and 550 genes
🔗 Matching reference genes in the model


🧬 477 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
/home/momo/miniforge3/envs/abca-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!


In [21]:
predictions.predicted_labels.head()

,predicted_labels,over_clustering,majority_voting
cell_label,,,
1017092617101530168-1,068 LSX Otx2 Gaba,273,068 LSX Otx2 Gaba
1017092617101600264-1,173 PAG Pou4f2 Glut,273,068 LSX Otx2 Gaba
1017092617101570130-1,068 LSX Otx2 Gaba,273,068 LSX Otx2 Gaba
1017092617101600254,053 Sst Gaba,273,068 LSX Otx2 Gaba
1017092617101290732,053 Sst Gaba,273,068 LSX Otx2 Gaba


In [23]:
# Load your dataframes (assuming they are named `predictions_df` and `metadata_df`)
merged_df = predictions.predicted_labels[["majority_voting"]].merge(adata_section.obs[['subclass']], left_index=True, right_index=True, how='inner')

# Display merged data
print(merged_df.head())

                         majority_voting           subclass
cell_label                                                 
1017092617101530168-1  068 LSX Otx2 Gaba  038 DG-PIR Ex IMN
1017092617101600264-1  068 LSX Otx2 Gaba  038 DG-PIR Ex IMN
1017092617101570130-1  068 LSX Otx2 Gaba  038 DG-PIR Ex IMN
1017092617101600254    068 LSX Otx2 Gaba  038 DG-PIR Ex IMN
1017092617101290732    068 LSX Otx2 Gaba  038 DG-PIR Ex IMN


In [25]:
accuracy = (merged_df['majority_voting'].astype(str) == merged_df['subclass'].astype(str)).mean()
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 31.16%
